# Smart Waste Classification - Google Colab GPU Training Notebook
**Project:** College Mini-Project (BTech Computer Engineering)
**Dataset:** TrashNet (Gary Thung & Mindy Yang, Stanford CS229)
**Output:** `waste_classifier.keras` and `class_indices.json`

> **Instructions:**
> 1. Go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.
> 2. Click **Runtime -> Run all**.
> 3. When training finishes, the notebook will download the trained `waste_classifier.keras` model.
> 4. Move that file into your project's `models/` directory.

In [ ]:
# Check GPU Availability
!nvidia-smi

In [ ]:
# Download and Unzip TrashNet Dataset
import os
import urllib.request
import zipfile

url = "https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip"
zip_dest = "dataset-resized.zip"

if not os.path.exists("dataset-resized"):
    print("Downloading TrashNet dataset (~41MB)...")
    urllib.request.urlretrieve(url, zip_dest)
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_dest, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extraction complete!")
else:
    print("Dataset folder already exists.")

print("Classes found:", sorted(os.listdir("dataset-resized")))

In [ ]:
# Import Dependencies
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)
print("GPU Devices Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Create Dataset Pipeline with Augmentation and Normalization
DATA_DIR = "dataset-resized"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# 80% Training Split
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True
)

# 20% Validation Split
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
print("Class Names:", class_names)

# Save Class Index Mapping
class_mapping = {idx: name for idx, name in enumerate(class_names)}
with open("class_indices.json", "w") as f:
    json.dump(class_mapping, f, indent=2)
print("Saved class_indices.json")

# Preprocessing & Data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])
rescale = layers.Rescaling(1.0 / 255.0)

train_ds = train_ds.map(lambda x, y: (rescale(data_augmentation(x, training=True)), y), num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (rescale(x), y), num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# Build CNN Architecture
def build_waste_cnn(input_shape=(224, 224, 3), num_classes=6):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        # Conv Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        # Conv Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        # Conv Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        # Dense Classifier
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_waste_cnn(num_classes=len(class_names))
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Train CNN with Early Stopping & Model Checkpoint
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(filepath="waste_classifier.keras", monitor="val_accuracy", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks
)

In [ ]:
# Plot Training Accuracy & Loss Curves
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Acc', color='#2e7d32')
plt.plot(epochs_range, val_acc, label='Val Acc', color='#1565c0', linestyle='--')
plt.title('Accuracy')
plt.legend()
plt.grid(True, linestyle=':')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss', color='#c62828')
plt.plot(epochs_range, val_loss, label='Val Loss', color='#e65100', linestyle='--')
plt.title('Loss')
plt.legend()
plt.grid(True, linestyle=':')
plt.savefig('training_history.png', dpi=200)
plt.show()

In [ ]:
# Evaluate Classification Report and Confusion Matrix
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
plt.imshow(cm, cmap=plt.cm.Greens)
plt.title('Confusion Matrix')
plt.colorbar()
ticks = np.arange(len(class_names))
plt.xticks(ticks, class_names, rotation=45)
plt.yticks(ticks, class_names)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=200)
plt.show()

In [ ]:
# Download the trained model artifacts to your local computer
from google.colab import files

print("Downloading trained model weights and class index mapping...")
files.download('waste_classifier.keras')
files.download('class_indices.json')
print("Place these two files into your local project's 'models/' folder!")